In [64]:
# 🚀 1. Inicialização do Spark
import os
import sys

os.environ['PYSPARK_PYTHON'] = "/Users/lpasch/sources/spark-3.4.3/python:/Users/lpasch/sources/spark-3.4.3/python/lib/py4j-0.10.9.7-src.zip:/Users/lpasch/sources/spark-3.4.3/python:/Users/lpasch/sources/spark-3.4.3/python/lib/py4j-0.10.9.7-src.zip:/Users/lpasch/sources/spark-3.4.3/python:/Users/lpasch/sources/spark-3.4.3/python/lib/py4j-0.10.9.7-src.zip:"
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['JAVA_HOME'] = "/opt/homebrew/Cellar/openjdk@11/11.0.31"

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkSQL-Kafka-Postgres") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-avro_2.12:3.5.0") \
    .getOrCreate()


In [65]:
# ✅ 2. Leitura do PostgreSQL com tabela e query reais

df_pg = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("driver", "org.postgresql.Driver") \
    .option("user", "postgres") \
    .option("password", "password") \
    .option("dbtable", "(SELECT \"CompraManha\", \"VendaManha\", \"PUCompraManha\", \"PUVendaManha\", \"PUBaseManha\", \"Data_Vencimento\", \"Data_Base\", \"Tipo\", dt_update FROM public.tesouro_ipca) AS dados") \
    .load()

df_pg.show()
df_pg.createOrReplaceTempView("tabela_ipca")


+-----------+----------+-------------+------------+-----------+-------------------+-------------------+----+--------------------+
|CompraManha|VendaManha|PUCompraManha|PUVendaManha|PUBaseManha|    Data_Vencimento|          Data_Base|Tipo|           dt_update|
+-----------+----------+-------------+------------+-----------+-------------------+-------------------+----+--------------------+
|       6.07|      6.15|       915.67|      906.48|     906.12|2024-08-15 00:00:00|2011-03-17 00:00:00|IPCA|2026-06-21 20:15:...|
|       5.79|      5.89|       519.12|      507.44|     507.25|2035-05-15 00:00:00|2011-03-17 00:00:00|IPCA|2026-06-21 20:15:...|
|       6.06|      6.14|       916.46|      907.26|      906.9|2024-08-15 00:00:00|2011-03-16 00:00:00|IPCA|2026-06-21 20:15:...|
|       6.55|      6.59|      1546.09|     1543.68|    1543.04|2015-05-15 00:00:00|2011-03-16 00:00:00|IPCA|2026-06-21 20:15:...|
|       5.78|      5.88|        520.1|       508.4|      508.2|2035-05-15 00:00:00|2011-03

In [66]:
# 📊 3. Consulta SQL sobre dados do PostgreSQL
spark.sql("""
    SELECT Tipo, COUNT(*) AS total
    FROM tabela_ipca
    GROUP BY Tipo
""").show()


+-----+------+
| Tipo| total|
+-----+------+
| IPCA|220920|
|IPCA+|     1|
+-----+------+



In [67]:
# 📦 4. Leitura do Kafka
# Certifique-se de que o tópico e o bootstrap server estão corretos

df_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "tesouro_ipca") \
    .load()

# Convertendo o valor em string
df_kafka_str = df_kafka.selectExpr("CAST(value AS STRING)")
#df_kafka_str = df_kafka.selectExpr("value")

df_kafka_str.show()
df_kafka_str.createOrReplaceTempView("mensagens_kafka")


+--------------------+
|               value|
+--------------------+
|{"CompraManha":6....|
|{"CompraManha":5....|
|{"CompraManha":6....|
|{"CompraManha":6....|
|{"CompraManha":5....|
|{"CompraManha":6....|
|{"CompraManha":6....|
|{"CompraManha":5....|
|{"CompraManha":5....|
|{"CompraManha":6....|
|{"CompraManha":5....|
|{"CompraManha":5....|
|{"CompraManha":5....|
|{"CompraManha":6....|
|{"CompraManha":5....|
|{"CompraManha":5....|
|{"CompraManha":6....|
|{"CompraManha":6....|
|{"CompraManha":6....|
|{"CompraManha":6....|
+--------------------+
only showing top 20 rows



26/06/21 17:15:23 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [68]:
# 🔍 5. Consulta SQL sobre dados do Kafka
spark.sql("""
    SELECT value, LENGTH(value) as tamanho
    FROM mensagens_kafka
""").show()


+--------------------+-------+
|               value|tamanho|
+--------------------+-------+
|{"CompraManha":6....|    202|
|{"CompraManha":5....|    202|
|{"CompraManha":6....|    201|
|{"CompraManha":6....|    205|
|{"CompraManha":5....|    199|
|{"CompraManha":6....|    202|
|{"CompraManha":6....|    205|
|{"CompraManha":5....|    202|
|{"CompraManha":5....|    201|
|{"CompraManha":6....|    205|
|{"CompraManha":5....|    202|
|{"CompraManha":5....|    202|
|{"CompraManha":5....|    200|
|{"CompraManha":6....|    205|
|{"CompraManha":5....|    200|
|{"CompraManha":5....|    201|
|{"CompraManha":6....|    205|
|{"CompraManha":6....|    202|
|{"CompraManha":6....|    201|
|{"CompraManha":6....|    205|
+--------------------+-------+
only showing top 20 rows



26/06/21 17:15:23 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [69]:
# 🛑 6. Encerramento
spark.stop()
